# Day 46 · Softmax and temperature: from scores to probabilities

**First-principles question:** What does the 'temperature' setting really do?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import urllib.request
import numpy as np
import matplotlib.pyplot as plt

def download(url, name):
    """Download once into data/downloads/ (ignored by git), then reuse."""
    path = ROOT / "data" / "downloads" / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path.read_text(encoding="utf-8")

words = download("https://raw.githubusercontent.com/karpathy/makemore/master/names.txt", "names.txt").split()
chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}; stoi["."] = 0      # "." marks the start and the end of a name
itos = {i: ch for ch, i in stoi.items()}
print(len(words), "names, e.g.", words[:5])
N = np.zeros((27, 27), dtype=np.int64)
for w in words:
    seq = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(seq, seq[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

## 1 · Softmax from scratch

In [ ]:
def softmax(scores, temperature=1.0):
    """Turn any scores into probabilities that add up to 1."""
    z = np.asarray(scores, dtype=float) / temperature
    # TODO: subtract z.max() (so e^z never overflows), take np.exp, divide by the sum
    ...

print(softmax([1.0, 2.0, 3.0]))
print(softmax([1000.0, 1001.0, 1002.0]))      # the naive version overflows here

## 2 · Temperature on one row
The log of the counts are 'scores' for the next letter after 'a'.

In [ ]:
scores = np.log(N[stoi["a"]] + 1)
letters = [itos[i] for i in range(27)]
fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for ax, T in zip(axes, [0.2, 1.0, 2.0]):
    ax.bar(letters, softmax(scores, T), color="#2153c4"); ax.set_title(f"temperature {T}")
plt.suptitle("P(next letter | 'a')"); plt.show()

## 3 · Names at three temperatures

In [ ]:
rng = np.random.default_rng(42)
logits = np.log(N + 1)

def sample(T):
    out, ix = [], 0
    while True:
        ix = rng.choice(27, p=softmax(logits[ix], T))
        if ix == 0:
            return "".join(out)
        out.append(itos[ix])

for T in [0.2, 1.0, 2.0]:
    print(f"T={T}:", [sample(T) for _ in range(8)])

**How this explains your day 1 experiment (temperature 0.8 vs. 0):** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
p = softmax([1.0, 2.0, 3.0])
assert abs(p.sum() - 1) < 1e-12 and p[2] > p[1] > p[0]
assert np.allclose(softmax([1000.0, 1001.0, 1002.0]), p), "Subtract the max to survive big numbers"
assert np.allclose(softmax([1.0, 2.0, 3.0]), softmax([11.0, 12.0, 13.0])), "Adding a constant changes nothing"
assert softmax([1.0, 2.0, 3.0], 0.01)[2] > 0.999, "A tiny temperature picks the top score"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")